# Exploración de archivos XBRL — ISA

Este notebook tiene como objetivo explorar los estados financieros
reportados por ISA en formato XBRL y determinar:

1. La estructura de los archivos.
2. Los conceptos XBRL correspondientes a las variables financieras de interés.
3. Las unidades y fechas asociadas a cada dato.
4. El tratamiento necesario para convertir información YTD en datos trimestrales.

Las variables de interés son:

- Ingresos
- EBITDA
- Deuda corriente
- Deuda no corriente
- Gastos financieros
- Caja y equivalentes de efectivo

Este notebook corresponde a una etapa exploratoria. La extracción
automatizada y la construcción del panel definitivo se realizarán posteriormente.

In [1]:
import collections
import collections.abc

# Compatibilidad de Arelle con versiones modernas de Python
for _cls in [
    "MutableSet", "MutableMapping", "Mapping", "Sequence",
    "Callable", "Iterable", "Container", "Hashable",
    "Sized", "Set", "MutableSequence"
]:
    if not hasattr(collections, _cls) and hasattr(collections.abc, _cls):
        setattr(collections, _cls, getattr(collections.abc, _cls))

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd
import numpy as np

from arelle import Cntlr

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_ISA_PATH = PROJECT_ROOT / "Data" / "raw" / "ISA"

list(RAW_ISA_PATH.iterdir())[:10]

[WindowsPath('c:/Users/57316/Documents/SIMON MONSALVE/UNIVERSIDAD/OCTAVO SEMESTRE/credit_risk_estimations/credit-risk-colombia/Data/raw/ISA/2021Q1_2021-03-31.xbrl'),
 WindowsPath('c:/Users/57316/Documents/SIMON MONSALVE/UNIVERSIDAD/OCTAVO SEMESTRE/credit_risk_estimations/credit-risk-colombia/Data/raw/ISA/2021Q2_2021-06-30.xbrl'),
 WindowsPath('c:/Users/57316/Documents/SIMON MONSALVE/UNIVERSIDAD/OCTAVO SEMESTRE/credit_risk_estimations/credit-risk-colombia/Data/raw/ISA/2021Q3_2021-09-30.xbrl'),
 WindowsPath('c:/Users/57316/Documents/SIMON MONSALVE/UNIVERSIDAD/OCTAVO SEMESTRE/credit_risk_estimations/credit-risk-colombia/Data/raw/ISA/2021Q4_2021-12-31.xbrl')]

In [3]:
print("Directorio actual:", Path.cwd())
print("Ruta ISA:", RAW_ISA_PATH)

Directorio actual: c:\Users\57316\Documents\SIMON MONSALVE\UNIVERSIDAD\OCTAVO SEMESTRE\credit_risk_estimations\credit-risk-colombia\notebooks
Ruta ISA: c:\Users\57316\Documents\SIMON MONSALVE\UNIVERSIDAD\OCTAVO SEMESTRE\credit_risk_estimations\credit-risk-colombia\Data\raw\ISA


In [4]:
xbrl_files = sorted(RAW_ISA_PATH.glob("*.xbrl"))

print(f"Archivos encontrados: {len(xbrl_files)}")

for file in xbrl_files:
    print(file.name)

Archivos encontrados: 4
2021Q1_2021-03-31.xbrl
2021Q2_2021-06-30.xbrl
2021Q3_2021-09-30.xbrl
2021Q4_2021-12-31.xbrl


In [5]:
file_path = xbrl_files[0]

print(file_path)

cntlr = Cntlr.Cntlr()

model_xbrl = cntlr.modelManager.load(str(file_path))

print("Archivo cargado correctamente")
print("Número de facts:", len(list(model_xbrl.facts)))

c:\Users\57316\Documents\SIMON MONSALVE\UNIVERSIDAD\OCTAVO SEMESTRE\credit_risk_estimations\credit-risk-colombia\Data\raw\ISA\2021Q1_2021-03-31.xbrl
Archivo cargado correctamente
Número de facts: 6672


In [6]:
facts = []

for fact in model_xbrl.facts:

    context = fact.context

    facts.append({
        "name": str(fact.concept.qname) if fact.concept is not None else None,
        "value": fact.value,
        "isNumeric": fact.isNumeric,
        "contextID": fact.contextID,
        "unitID": fact.unitID,
        "startDate": context.startDatetime,
        "endDate": context.endDatetime,
        "isInstant": context.isInstantPeriod,
        "isStartEnd": context.isStartEndPeriod,
    })

facts_df = pd.DataFrame(facts)

facts_df.head()

,name,value,isNumeric,contextID,unitID,startDate,endDate,isInstant,isStartEnd
0,co-sfc-core:ActividadPrincipal,"""ISA tiene por objeto:\n\n?\tLa prestación del...",False,TrimestreAcumuladoActual,NaN,2021-01-01,2021-04-01,False,True
1,co-sfc-core:ActivosFinancierosCorrientesAlCost...,3437159486,True,CierreTrimestreActual,COP,NaT,2021-04-01,True,False
2,co-sfc-core:ActivosFinancierosCorrientesAlCost...,3781712973,True,SaldoActualInicio,COP,NaT,2021-01-01,True,False
3,co-sfc-core:ActivosVinculadosNegociosConjuntos,17179102866,True,CierreTrimestreActual,COP,NaT,2021-04-01,True,False
4,co-sfc-core:ActivosVinculadosNegociosConjuntos,16483412324,True,CierreTrimestreAnterior,COP,NaT,2020-04-01,True,False


In [7]:
# Conceptos candidatos que queremos inspeccionar
candidate_tags = [
    "ifrs:Revenue",
    "ifrs:RevenueAndOperatingIncome",
    "ifrs:Cash",
    "ifrs:CashAndCashEquivalents",
    "ifrs:CurrentBorrowingsAndCurrentPortionOfNoncurrentLoansReceived",
    "ifrs:ShorttermBorrowings",
    "ifrs:Borrowings",
    "ifrs:LongtermBorrowings",
    "ifrs:FinanceCosts",
    "ifrs:FinanceIncomeCost",
    "ifrs:InterestExpenseOnBorrowings",
    "ifrs:DepreciationAndAmortisationExpense",
    "ifrs:DepreciationExpense",
    "ifrs:AmortisationExpense"
]

candidate_facts = facts_df[
    facts_df["name"].isin(candidate_tags)
].copy()

candidate_facts.sort_values(
    ["name", "endDate"]
)

,name,value,isNumeric,contextID,unitID,startDate,endDate,isInstant,isStartEnd
628,ifrs:AmortisationExpense,97129867,True,TrimestreAcumuladoAnterior,COP,2020-01-01,2020-04-01,False,True
627,ifrs:AmortisationExpense,95162517,True,TrimestreAcumuladoActual,COP,2021-01-01,2021-04-01,False,True
689,ifrs:Borrowings,22468835935,True,SaldoActualInicio,COP,NaT,2021-01-01,True,False
688,ifrs:Borrowings,23971643898,True,CierreTrimestreActual,COP,NaT,2021-04-01,True,False
693,ifrs:Cash,918038388,True,SaldoActualInicio,COP,NaT,2021-01-01,True,False
692,ifrs:Cash,1346544460,True,CierreTrimestreActual,COP,NaT,2021-04-01,True,False
699,ifrs:CashAndCashEquivalents,2487202068,True,SaldoAnteriorInicio,COP,NaT,2020-01-01,True,False
327,ifrs:CashAndCashEquivalents,1407146108,True,Saldo_co-sfc-core_InformacionRevelarNegociosCo...,COP,NaT,2020-04-01,True,False
329,ifrs:CashAndCashEquivalents,1407146108,True,Saldo_co-sfc-core_InformacionRevelarNegociosCo...,COP,NaT,2020-04-01,True,False
331,ifrs:CashAndCashEquivalents,1407146108,True,Saldo_co-sfc-core_InformacionRevelarNegociosCo...,COP,NaT,2020-04-01,True,False


In [8]:
pd.set_option("display.max_rows", 200)

concepts = (
    facts_df["name"]
    .dropna()
    .drop_duplicates()
    .sort_values()
)

print(f"Número de conceptos únicos: {len(concepts)}")

concepts.to_frame().head(100)

concepts[
    concepts.str.contains(
        "Revenue|Income|Cash|Debt|Borrow|Interest|Operating|Depreciation|Amort",
        case=False,
        regex=True
    )
]

Número de conceptos únicos: 985


1       co-sfc-core:ActivosFinancierosCorrientesAlCost...
7              co-sfc-core:AjusteAlCostoAmortizadoCartera
10      co-sfc-core:AjusteAlCostoAmortizadoOtrosActivo...
28      co-sfc-core:AjusteCostoAmortizadoPasivosFinanc...
466              ifrs:AccumulatedOtherComprehensiveIncome
627                              ifrs:AmortisationExpense
629     ifrs:AmortisationIntangibleAssetsOtherThanGood...
688                                       ifrs:Borrowings
692                                             ifrs:Cash
694     ifrs:CashAdvancesAndLoansMadeToOtherPartiesCla...
326                           ifrs:CashAndCashEquivalents
700     ifrs:CashAndCashEquivalentsAmountContributedTo...
708                                  ifrs:CashEquivalents
710     ifrs:CashFlowsFromLosingControlOfSubsidiariesO...
712           ifrs:CashFlowsFromUsedInFinancingActivities
714           ifrs:CashFlowsFromUsedInInvestingActivities
716           ifrs:CashFlowsFromUsedInOperatingActivities
718           

## Identificación de conceptos financieros

A continuación se inspeccionan los conceptos XBRL potencialmente
relacionados con las variables requeridas para el análisis.

In [9]:
keywords = {
    "ingresos": "revenue|income",
    "caja": "cash|cash.*equivalent",
    "deuda": "debt|borrow|loan",
    "intereses": "interest|finance",
    "depreciacion": "depreciation",
    "amortizacion": "amortisation|amortization",
}

for variable, pattern in keywords.items():
    print(f"\n### {variable.upper()}")
    result = concepts[
        concepts.str.contains(pattern, case=False, regex=True, na=False)
    ]
    print(result.to_string(index=False))


### INGRESOS
          ifrs:AccumulatedOtherComprehensiveIncome
                          ifrs:ComprehensiveIncome
ifrs:ComprehensiveIncomeAttributableToNoncontro...
ifrs:ComprehensiveIncomeAttributableToOwnersOfP...
                      ifrs:CurrentTaxExpenseIncome
ifrs:CurrentTaxExpenseIncomeAndAdjustmentsForCu...
ifrs:DeferredTaxExpenseIncomeRecognisedInProfit...
ifrs:DeferredTaxExpenseIncomeRelatingToOriginat...
ifrs:DescriptionOfAccountingPolicyForFeeAndComm...
ifrs:DescriptionOfAccountingPolicyForFinanceInc...
ifrs:DescriptionOfAccountingPolicyForIncomeTaxE...
ifrs:DescriptionOfAccountingPolicyForInterestIn...
ifrs:DescriptionOfAccountingPolicyForRecognitio...
ifrs:DescriptionOfAccountingPolicyForTradingInc...
ifrs:DisclosureOfAnalysisOfOtherComprehensiveIn...
        ifrs:DisclosureOfDeferredIncomeExplanatory
ifrs:DisclosureOfFeeAndCommissionIncomeExpenseE...
  ifrs:DisclosureOfFinanceIncomeExpenseExplanatory
         ifrs:DisclosureOfFinanceIncomeExplanatory
             ifrs

In [10]:
for tag in candidate_tags:

    subset = facts_df[facts_df["name"] == tag].copy()

    if not subset.empty:
        print("\n" + "=" * 100)
        print(tag)
        display(
            subset[
                [
                    "name",
                    "value",
                    "contextID",
                    "unitID",
                    "startDate",
                    "endDate",
                    "isInstant",
                    "isStartEnd"
                ]
            ].sort_values(["endDate", "startDate"])
        )


ifrs:Revenue


,name,value,contextID,unitID,startDate,endDate,isInstant,isStartEnd
441,ifrs:Revenue,2190069478,Periodo_co-sfc-core_InformacionRevelarNegocios...,COP,2020-01-01,2020-04-01,False,True
443,ifrs:Revenue,2190069478,Periodo_co-sfc-core_InformacionRevelarNegocios...,COP,2020-01-01,2020-04-01,False,True
445,ifrs:Revenue,2190069478,Periodo_co-sfc-core_InformacionRevelarNegocios...,COP,2020-01-01,2020-04-01,False,True
6538,ifrs:Revenue,2069572769,TrimestreAcumuladoAnterior,COP,2020-01-01,2020-04-01,False,True
6533,ifrs:Revenue,9110846607.0000,AnualAnterior_ifrs_DisclosureOfSignificantInve...,COP,2020-01-01,2021-01-01,False,True
6534,ifrs:Revenue,9110846607,AnualAnterior_ifrs_DisclosureOfSignificantInve...,COP,2020-01-01,2021-01-01,False,True
440,ifrs:Revenue,542724541,Periodo_co-sfc-core_InformacionRevelarNegocios...,COP,2021-01-01,2021-04-01,False,True
442,ifrs:Revenue,542724541,Periodo_co-sfc-core_InformacionRevelarNegocios...,COP,2021-01-01,2021-04-01,False,True
444,ifrs:Revenue,542724541,Periodo_co-sfc-core_InformacionRevelarNegocios...,COP,2021-01-01,2021-04-01,False,True
6535,ifrs:Revenue,2365046371.0000,Periodo_ifrs_DisclosureOfSignificantInvestment...,COP,2021-01-01,2021-04-01,False,True



ifrs:RevenueAndOperatingIncome


,name,value,contextID,unitID,startDate,endDate,isInstant,isStartEnd
6540,ifrs:RevenueAndOperatingIncome,2069572769,TrimestreAcumuladoAnterior,COP,2020-01-01,2020-04-01,False,True
6539,ifrs:RevenueAndOperatingIncome,2365046371,TrimestreAcumuladoActual,COP,2021-01-01,2021-04-01,False,True



ifrs:Cash


...

,name,value,contextID,unitID,startDate,endDate,isInstant,isStartEnd
359,ifrs:DepreciationAndAmortisationExpense,37491975,Periodo_co-sfc-core_InformacionRevelarNegocios...,COP,2020-01-01,2020-04-01,False,True
361,ifrs:DepreciationAndAmortisationExpense,37491975,Periodo_co-sfc-core_InformacionRevelarNegocios...,COP,2020-01-01,2020-04-01,False,True
363,ifrs:DepreciationAndAmortisationExpense,37491975,Periodo_co-sfc-core_InformacionRevelarNegocios...,COP,2020-01-01,2020-04-01,False,True
1326,ifrs:DepreciationAndAmortisationExpense,207040554,TrimestreAcumuladoAnterior,COP,2020-01-01,2020-04-01,False,True
358,ifrs:DepreciationAndAmortisationExpense,7910634,Periodo_co-sfc-core_InformacionRevelarNegocios...,COP,2021-01-01,2021-04-01,False,True
360,ifrs:DepreciationAndAmortisationExpense,7910634,Periodo_co-sfc-core_InformacionRevelarNegocios...,COP,2021-01-01,2021-04-01,False,True
362,ifrs:DepreciationAndAmortisationExpense,7910634,Periodo_co-sfc-core_InformacionRevelarNegocios...,COP,2021-01-01,2021-04-01,False,True
1325,ifrs:DepreciationAndAmortisationExpense,211194859,TrimestreAcumuladoActual,COP,2021-01-01,2021-04-01,False,True



ifrs:DepreciationExpense


,name,value,contextID,unitID,startDate,endDate,isInstant,isStartEnd
1328,ifrs:DepreciationExpense,109910687,TrimestreAcumuladoAnterior,COP,2020-01-01,2020-04-01,False,True
1327,ifrs:DepreciationExpense,116032342,TrimestreAcumuladoActual,COP,2021-01-01,2021-04-01,False,True



ifrs:AmortisationExpense


,name,value,contextID,unitID,startDate,endDate,isInstant,isStartEnd
628,ifrs:AmortisationExpense,97129867,TrimestreAcumuladoAnterior,COP,2020-01-01,2020-04-01,False,True
627,ifrs:AmortisationExpense,95162517,TrimestreAcumuladoActual,COP,2021-01-01,2021-04-01,False,True


In [11]:
contexts = []

for context_id, context in model_xbrl.contexts.items():

    contexts.append({
        "contextID": context_id,
        "startDate": context.startDatetime,
        "endDate": context.endDatetime,
        "isInstant": context.isInstantPeriod,
        "isStartEnd": context.isStartEndPeriod,
    })

contexts_df = pd.DataFrame(contexts)

contexts_df.sort_values(
    ["endDate", "startDate"]
).head(50)

,contextID,startDate,endDate,isInstant,isStartEnd
50,AnualPrevioAnterior_ifrs_DisclosureOfComparati...,2019-01-01,2020-01-01,False,True
51,AnualPrevioAnterior_ifrs_DisclosureOfComparati...,2019-01-01,2020-01-01,False,True
52,AnualPrevioAnterior_ifrs_DisclosureOfComparati...,2019-01-01,2020-01-01,False,True
994,SaldoAnteriorInicio,NaT,2020-01-01,True,False
1009,SaldoInicio_ifrs_DisclosureOfClassesOfShareCap...,NaT,2020-01-01,True,False
1011,SaldoInicio_ifrs_DisclosureOfClassesOfShareCap...,NaT,2020-01-01,True,False
1012,SaldoInicio_ifrs_DisclosureOfComparativeInform...,NaT,2020-01-01,True,False
1013,SaldoInicio_ifrs_DisclosureOfComparativeInform...,NaT,2020-01-01,True,False
1014,SaldoInicio_ifrs_DisclosureOfComparativeInform...,NaT,2020-01-01,True,False
1025,SaldoInicio_ifrs_DisclosureOfIntangibleAssetsT...,NaT,2020-01-01,True,False


In [12]:
contexts_df[
    contexts_df["endDate"].dt.year == 2021
].sort_values(
    ["endDate", "startDate"]
)

,contextID,startDate,endDate,isInstant,isStartEnd
0,AnualAnterior_co-sfc-core_InformacionRevelarAc...,2020-01-01,2021-01-01,False,True
1,AnualAnterior_ifrs_DisclosureOfClassesOfShareC...,2020-01-01,2021-01-01,False,True
2,AnualAnterior_ifrs_DisclosureOfClassesOfShareC...,2020-01-01,2021-01-01,False,True
3,AnualAnterior_ifrs_DisclosureOfClassesOfShareC...,2020-01-01,2021-01-01,False,True
4,AnualAnterior_ifrs_DisclosureOfHedgeAccounting...,2020-01-01,2021-01-01,False,True
...,...,...,...,...,...
983,Saldo_ifrs_StatementOfChangesInEquityTable_Tre...,NaT,2021-04-01,True,False
985,Saldo_ifrs_StatementOfChangesInEquityTable_Tre...,NaT,2021-04-01,True,False
987,Saldo_ifrs_StatementOfChangesInEquityTable_Tre...,NaT,2021-04-01,True,False
989,Saldo_ifrs_StatementOfChangesInEquityTable_Tre...,NaT,2021-04-01,True,False


## Validación de conceptos financieros

In [13]:
validation_tags = {
    "Ingresos": "ifrs:Revenue",
    "Caja": "ifrs:CashAndCashEquivalents",
    "Deuda CP": "ifrs:ShorttermBorrowings",
    "Deuda LP": "ifrs:LongtermBorrowings",
    "Deuda Total": "ifrs:Borrowings",
    "Gastos financieros": "ifrs:FinanceCosts",
    "Intereses deuda": "ifrs:InterestExpenseOnBorrowings",
    "Resultado operativo": "ifrs:ProfitLossFromOperatingActivities",
    "Depreciación": "ifrs:DepreciationExpense",
    "Amortización": "ifrs:AmortisationExpense",
    "D&A": "ifrs:DepreciationAndAmortisationExpense",
}

validation_rows = []

for variable, tag in validation_tags.items():

    subset = facts_df[facts_df["name"] == tag].copy()

    for _, row in subset.iterrows():
        validation_rows.append({
            "variable": variable,
            "tag": tag,
            "value": row["value"],
            "unitID": row["unitID"],
            "contextID": row["contextID"],
            "startDate": row["startDate"],
            "endDate": row["endDate"],
            "isInstant": row["isInstant"],
            "isStartEnd": row["isStartEnd"],
        })

validation_df = pd.DataFrame(validation_rows)

validation_df.sort_values(
    ["variable", "endDate", "startDate"]
)

,variable,tag,value,unitID,contextID,startDate,endDate,isInstant,isStartEnd
37,Amortización,ifrs:AmortisationExpense,97129867,COP,TrimestreAcumuladoAnterior,2020-01-01,2020-04-01,False,True
36,Amortización,ifrs:AmortisationExpense,95162517,COP,TrimestreAcumuladoActual,2021-01-01,2021-04-01,False,True
21,Caja,ifrs:CashAndCashEquivalents,2487202068,COP,SaldoAnteriorInicio,NaT,2020-01-01,True,False
13,Caja,ifrs:CashAndCashEquivalents,1407146108,COP,Saldo_co-sfc-core_InformacionRevelarNegociosCo...,NaT,2020-04-01,True,False
15,Caja,ifrs:CashAndCashEquivalents,1407146108,COP,Saldo_co-sfc-core_InformacionRevelarNegociosCo...,NaT,2020-04-01,True,False
17,Caja,ifrs:CashAndCashEquivalents,1407146108,COP,Saldo_co-sfc-core_InformacionRevelarNegociosCo...,NaT,2020-04-01,True,False
19,Caja,ifrs:CashAndCashEquivalents,2797800540,COP,CierreTrimestreAnterior,NaT,2020-04-01,True,False
20,Caja,ifrs:CashAndCashEquivalents,3781712973,COP,SaldoActualInicio,NaT,2021-01-01,True,False
12,Caja,ifrs:CashAndCashEquivalents,1249809982,COP,Saldo_co-sfc-core_InformacionRevelarNegociosCo...,NaT,2021-04-01,True,False
14,Caja,ifrs:CashAndCashEquivalents,1249809982,COP,Saldo_co-sfc-core_InformacionRevelarNegociosCo...,NaT,2021-04-01,True,False


## Validación detallada del archivo 2021Q1

Se inspeccionan únicamente los once tags solicitados. La consolidación se identifica por la ausencia de dimensiones XBRL en el contexto; los facts dimensionales se conservan para detectar duplicidades, pero no se mezclan con el fact consolidado.

Arelle representa el extremo final del intervalo como fecha exclusiva: `2021-04-01` corresponde al trimestre cerrado el 31 de marzo de 2021.

In [14]:
validation_tags_exact = [
    "ifrs:Revenue",
    "ifrs:CashAndCashEquivalents",
    "ifrs:ShorttermBorrowings",
    "ifrs:LongtermBorrowings",
    "ifrs:Borrowings",
    "ifrs:FinanceCosts",
    "ifrs:InterestExpenseOnBorrowings",
    "ifrs:ProfitLossFromOperatingActivities",
    "ifrs:DepreciationExpense",
    "ifrs:AmortisationExpense",
    "ifrs:DepreciationAndAmortisationExpense",
]

q1_path = RAW_ISA_PATH / "2021Q1_2021-03-31.xbrl"
q1_model = cntlr.modelManager.load(str(q1_path))

variable_by_tag = {
    "ifrs:Revenue": "ingresos",
    "ifrs:CashAndCashEquivalents": "caja",
    "ifrs:ShorttermBorrowings": "deuda_cp",
    "ifrs:LongtermBorrowings": "deuda_lp",
    "ifrs:Borrowings": "deuda_total",
    "ifrs:FinanceCosts": "gastos_financieros",
    "ifrs:InterestExpenseOnBorrowings": "intereses_deuda",
    "ifrs:ProfitLossFromOperatingActivities": "resultado_operativo",
    "ifrs:DepreciationExpense": "depreciacion",
    "ifrs:AmortisationExpense": "amortizacion",
    "ifrs:DepreciationAndAmortisationExpense": "depreciacion_amortizacion",
}

def fact_unit(fact):
    if fact.unit is None:
        return fact.unitID
    numerator, denominator = fact.unit.measures
    numerator_text = " * ".join(str(measure) for measure in numerator)
    denominator_text = " * ".join(str(measure) for measure in denominator)
    return numerator_text if not denominator_text else f"{numerator_text} / {denominator_text}"

def context_dimensions(context):
    return "; ".join(
        f"{dimension}={dimension_value.memberQname}"
        for dimension, dimension_value in context.qnameDims.items()
    )

detail_rows = []
for fact in q1_model.facts:
    tag = str(fact.concept.qname) if fact.concept is not None else None
    if tag not in validation_tags_exact:
        continue
    context = fact.context
    dimensions = context_dimensions(context)
    detail_rows.append({
        "variable": variable_by_tag[tag],
        "tag": tag,
        "valor": fact.value,
        "unitID": fact.unitID,
        "unidad": fact_unit(fact),
        "contextID": fact.contextID,
        "fecha_inicial": context.startDatetime,
        "fecha_final": context.endDatetime,
        "tipo": "instantaneo" if context.isInstantPeriod else "duracion",
        "dimensiones": dimensions or "sin dimensiones",
        "consolidado": not bool(dimensions),
        "ytd": (
            not context.isInstantPeriod
            and not dimensions
            and context.startDatetime is not None
            and context.startDatetime.month == 1
            and context.startDatetime.day == 1
        ),
    })

q1_facts_df = (
    pd.DataFrame(detail_rows)
    .sort_values(["tag", "fecha_final", "contextID"])
    .reset_index(drop=True)
)
display(q1_facts_df)
print(f"Facts encontrados: {len(q1_facts_df)}")

,variable,tag,valor,unitID,unidad,contextID,fecha_inicial,fecha_final,tipo,dimensiones,consolidado,ytd
0,amortizacion,ifrs:AmortisationExpense,97129867,COP,iso4217:COP,TrimestreAcumuladoAnterior,2020-01-01,2020-04-01,duracion,sin dimensiones,True,True
1,amortizacion,ifrs:AmortisationExpense,95162517,COP,iso4217:COP,TrimestreAcumuladoActual,2021-01-01,2021-04-01,duracion,sin dimensiones,True,True
2,deuda_total,ifrs:Borrowings,22468835935,COP,iso4217:COP,SaldoActualInicio,NaT,2021-01-01,instantaneo,sin dimensiones,True,False
3,deuda_total,ifrs:Borrowings,23971643898,COP,iso4217:COP,CierreTrimestreActual,NaT,2021-04-01,instantaneo,sin dimensiones,True,False
4,caja,ifrs:CashAndCashEquivalents,2487202068,COP,iso4217:COP,SaldoAnteriorInicio,NaT,2020-01-01,instantaneo,sin dimensiones,True,False
5,caja,ifrs:CashAndCashEquivalents,2797800540,COP,iso4217:COP,CierreTrimestreAnterior,NaT,2020-04-01,instantaneo,sin dimensiones,True,False
6,caja,ifrs:CashAndCashEquivalents,1407146108,COP,iso4217:COP,Saldo_co-sfc-core_InformacionRevelarNegociosCo...,NaT,2020-04-01,instantaneo,co-sfc-core:InformacionRevelarNegociosConjunto...,False,False
7,caja,ifrs:CashAndCashEquivalents,1407146108,COP,iso4217:COP,Saldo_co-sfc-core_InformacionRevelarNegociosCo...,NaT,2020-04-01,instantaneo,co-sfc-core:InformacionRevelarNegociosConjunto...,False,False
8,caja,ifrs:CashAndCashEquivalents,1407146108,COP,iso4217:COP,Saldo_co-sfc-core_InformacionRevelarNegociosCo...,NaT,2020-04-01,instantaneo,co-sfc-core:InformacionRevelarNegociosConjunto...,False,False
9,caja,ifrs:CashAndCashEquivalents,3781712973,COP,iso4217:COP,SaldoActualInicio,NaT,2021-01-01,instantaneo,sin dimensiones,True,False


Facts encontrados: 46


In [15]:
def one_consolidated_value(tag, context_id):
    values = q1_facts_df.loc[
        (q1_facts_df["tag"] == tag)
        & (q1_facts_df["contextID"] == context_id)
        & q1_facts_df["consolidado"],
        "valor",
    ]
    return float(values.iloc[0]) if len(values) == 1 else None

def count_consolidated(tag, context_id):
    return len(q1_facts_df.loc[
        (q1_facts_df["tag"] == tag)
        & (q1_facts_df["contextID"] == context_id)
        & q1_facts_df["consolidado"]
    ])

revenue = one_consolidated_value("ifrs:Revenue", "TrimestreAcumuladoActual")
finance_costs = one_consolidated_value("ifrs:FinanceCosts", "TrimestreAcumuladoActual")
shortterm_debt = one_consolidated_value("ifrs:ShorttermBorrowings", "CierreTrimestreActual")
longterm_debt = one_consolidated_value("ifrs:LongtermBorrowings", "CierreTrimestreActual")
borrowings = one_consolidated_value("ifrs:Borrowings", "CierreTrimestreActual")
operating_profit = one_consolidated_value("ifrs:ProfitLossFromOperatingActivities", "TrimestreAcumuladoActual")
depreciation = one_consolidated_value("ifrs:DepreciationExpense", "TrimestreAcumuladoActual")
amortisation = one_consolidated_value("ifrs:AmortisationExpense", "TrimestreAcumuladoActual")
depreciation_amortisation = one_consolidated_value("ifrs:DepreciationAndAmortisationExpense", "TrimestreAcumuladoActual")

ebitda_separate = (
    operating_profit + depreciation + amortisation
    if None not in [operating_profit, depreciation, amortisation]
    else None
)
ebitda_aggregate = (
    operating_profit + depreciation_amortisation
    if None not in [operating_profit, depreciation_amortisation]
    else None
)

summary_rows = []
for _, row in q1_facts_df[q1_facts_df["consolidado"]].iterrows():
    if row["contextID"] not in ["TrimestreAcumuladoActual", "CierreTrimestreActual"]:
        continue
    summary_rows.append({
        "variable": row["variable"],
        "tag": row["tag"],
        "valor": row["valor"],
        "tipo": row["tipo"],
        "contexto": row["contextID"],
        "conclusion": "flujo YTD consolidado" if row["ytd"] else "stock instantaneo consolidado" if row["tipo"] == "instantaneo" else "flujo consolidado",
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

print("1. Consolidación: se consideran consolidados los facts sin dimensiones; los facts dimensionales aparecen separados en q1_facts_df.")
print("2. Flujos YTD: Revenue, FinanceCosts, InterestExpenseOnBorrowings, ProfitLossFromOperatingActivities, DepreciationExpense, AmortisationExpense y DepreciationAndAmortisationExpense.")
print("   Stocks instantáneos: CashAndCashEquivalents, ShorttermBorrowings, LongtermBorrowings y Borrowings.")
print(f"3. Revenue: exactamente un consolidado YTD = {count_consolidated('ifrs:Revenue', 'TrimestreAcumuladoActual') == 1}.")
print(f"   FinanceCosts: exactamente un consolidado YTD = {count_consolidated('ifrs:FinanceCosts', 'TrimestreAcumuladoActual') == 1}.")
print(f"4. ShorttermBorrowings y LongtermBorrowings: exactamente un saldo consolidado al cierre = {shortterm_debt is not None and longterm_debt is not None}.")
print(f"5. Borrowings = ShorttermBorrowings + LongtermBorrowings = {borrowings == shortterm_debt + longterm_debt} ({borrowings} = {shortterm_debt} + {longterm_debt}).")
print(f"6. EBITDA separado = {ebitda_separate}; EBITDA agregado = {ebitda_aggregate}; mismo resultado = {ebitda_separate == ebitda_aggregate}.")
print("7. Inconsistencias: Revenue y CashAndCashEquivalents incluyen facts dimensionales; no deben sumarse al fact consolidado sin dimensiones.")

,variable,tag,valor,tipo,contexto,conclusion
0,amortizacion,ifrs:AmortisationExpense,95162517,duracion,TrimestreAcumuladoActual,flujo YTD consolidado
1,deuda_total,ifrs:Borrowings,23971643898,instantaneo,CierreTrimestreActual,stock instantaneo consolidado
2,caja,ifrs:CashAndCashEquivalents,3437159486,instantaneo,CierreTrimestreActual,stock instantaneo consolidado
3,depreciacion_amortizacion,ifrs:DepreciationAndAmortisationExpense,211194859,duracion,TrimestreAcumuladoActual,flujo YTD consolidado
4,depreciacion,ifrs:DepreciationExpense,116032342,duracion,TrimestreAcumuladoActual,flujo YTD consolidado
5,gastos_financieros,ifrs:FinanceCosts,352214757,duracion,TrimestreAcumuladoActual,flujo YTD consolidado
6,intereses_deuda,ifrs:InterestExpenseOnBorrowings,78781172,duracion,TrimestreAcumuladoActual,flujo YTD consolidado
7,deuda_lp,ifrs:LongtermBorrowings,22385117413,instantaneo,CierreTrimestreActual,stock instantaneo consolidado
8,resultado_operativo,ifrs:ProfitLossFromOperatingActivities,1286292845,duracion,TrimestreAcumuladoActual,flujo YTD consolidado
9,ingresos,ifrs:Revenue,2365046371,duracion,TrimestreAcumuladoActual,flujo YTD consolidado


1. Consolidación: se consideran consolidados los facts sin dimensiones; los facts dimensionales aparecen separados en q1_facts_df.
2. Flujos YTD: Revenue, FinanceCosts, InterestExpenseOnBorrowings, ProfitLossFromOperatingActivities, DepreciationExpense, AmortisationExpense y DepreciationAndAmortisationExpense.
   Stocks instantáneos: CashAndCashEquivalents, ShorttermBorrowings, LongtermBorrowings y Borrowings.
3. Revenue: exactamente un consolidado YTD = True.
   FinanceCosts: exactamente un consolidado YTD = True.
4. ShorttermBorrowings y LongtermBorrowings: exactamente un saldo consolidado al cierre = True.
5. Borrowings = ShorttermBorrowings + LongtermBorrowings = True (23971643898.0 = 1586526485.0 + 22385117413.0).
6. EBITDA separado = 1497487704.0; EBITDA agregado = 1497487704.0; mismo resultado = True.
7. Inconsistencias: Revenue y CashAndCashEquivalents incluyen facts dimensionales; no deben sumarse al fact consolidado sin dimensiones.


## Validación temporal de conceptos — ISA 2021

Se reutilizan los mismos once tags validados en Q1. Los flujos se seleccionan desde el contexto consolidado YTD actual y los stocks desde el contexto consolidado instantáneo de cierre.

In [16]:
files_2021 = sorted(RAW_ISA_PATH.glob("2021Q*.xbrl"))
print("Archivos 2021:")
for file in files_2021:
    print(file.name)

flow_tags = [
    "ifrs:Revenue",
    "ifrs:FinanceCosts",
    "ifrs:InterestExpenseOnBorrowings",
    "ifrs:ProfitLossFromOperatingActivities",
    "ifrs:DepreciationExpense",
    "ifrs:AmortisationExpense",
    "ifrs:DepreciationAndAmortisationExpense",
]
stock_tags = [
    "ifrs:CashAndCashEquivalents",
    "ifrs:ShorttermBorrowings",
    "ifrs:LongtermBorrowings",
    "ifrs:Borrowings",
]
all_temporal_tags = validation_tags_exact

period_rows = []
validation_counts = []

for file in files_2021:
    period = file.name.split("_")[0]
    period_model = cntlr.modelManager.load(str(file))
    facts_by_tag = {tag: [] for tag in all_temporal_tags}

    for fact in period_model.facts:
        tag = str(fact.concept.qname) if fact.concept is not None else None
        if tag not in facts_by_tag:
            continue
        context = fact.context
        dimensions = context_dimensions(context)
        if dimensions:
            continue
        facts_by_tag[tag].append({
            "tag": tag,
            "valor": float(fact.value),
            "unitID": fact.unitID,
            "contextID": fact.contextID,
            "fecha_inicial": context.startDatetime,
            "fecha_final": context.endDatetime,
            "instantaneo": context.isInstantPeriod,
        })

    for tag in all_temporal_tags:
        current_candidates = [
            row for row in facts_by_tag[tag]
            if row["contextID"] in ["TrimestreAcumuladoActual", "CierreTrimestreActual"]
        ]
        if tag in flow_tags:
            selected = [
                row for row in current_candidates
                if not row["instantaneo"]
                and row["fecha_inicial"] is not None
                and row["fecha_inicial"].month == 1
                and row["fecha_inicial"].day == 1
            ]
        else:
            selected = [
                row for row in current_candidates
                if row["instantaneo"]
                and row["contextID"] == "CierreTrimestreActual"
            ]
        validation_counts.append({
            "periodo": period,
            "tag": tag,
            "candidatos_consolidados": len(selected),
        })
        if len(selected) == 1:
            row = selected[0].copy()
            row["periodo"] = period
            period_rows.append(row)

    cntlr.modelManager.close(period_model)

validation_counts_df = pd.DataFrame(validation_counts)
temporal_facts_df = pd.DataFrame(period_rows)

print("Conteos de candidatos consolidados seleccionables:")
display(validation_counts_df.pivot(index="periodo", columns="tag", values="candidatos_consolidados"))
print("Facts seleccionados:")
display(temporal_facts_df.sort_values(["periodo", "tag"]))

Archivos 2021:
2021Q1_2021-03-31.xbrl
2021Q2_2021-06-30.xbrl
2021Q3_2021-09-30.xbrl
2021Q4_2021-12-31.xbrl
Conteos de candidatos consolidados seleccionables:


tag,ifrs:AmortisationExpense,ifrs:Borrowings,ifrs:CashAndCashEquivalents,ifrs:DepreciationAndAmortisationExpense,ifrs:DepreciationExpense,ifrs:FinanceCosts,ifrs:InterestExpenseOnBorrowings,ifrs:LongtermBorrowings,ifrs:ProfitLossFromOperatingActivities,ifrs:Revenue,ifrs:ShorttermBorrowings
periodo,,,,,,,,,,,
2021Q1,1,1,1,1,1,1,1,1,1,1,1
2021Q2,1,1,1,1,1,1,1,1,1,1,1
2021Q3,1,1,1,1,1,1,1,1,1,1,1
2021Q4,1,1,1,1,1,1,1,1,1,1,1


Facts seleccionados:


,tag,valor,unitID,contextID,fecha_inicial,fecha_final,instantaneo,periodo
9,ifrs:AmortisationExpense,9.516252e+07,COP,TrimestreAcumuladoActual,2021-01-01,2021-04-01,False,2021Q1
4,ifrs:Borrowings,2.397164e+10,COP,CierreTrimestreActual,NaT,2021-04-01,True,2021Q1
1,ifrs:CashAndCashEquivalents,3.437159e+09,COP,CierreTrimestreActual,NaT,2021-04-01,True,2021Q1
10,ifrs:DepreciationAndAmortisationExpense,2.111949e+08,COP,TrimestreAcumuladoActual,2021-01-01,2021-04-01,False,2021Q1
8,ifrs:DepreciationExpense,1.160323e+08,COP,TrimestreAcumuladoActual,2021-01-01,2021-04-01,False,2021Q1
5,ifrs:FinanceCosts,3.522148e+08,COP,TrimestreAcumuladoActual,2021-01-01,2021-04-01,False,2021Q1
6,ifrs:InterestExpenseOnBorrowings,7.878117e+07,COP,TrimestreAcumuladoActual,2021-01-01,2021-04-01,False,2021Q1
3,ifrs:LongtermBorrowings,2.238512e+10,COP,CierreTrimestreActual,NaT,2021-04-01,True,2021Q1
7,ifrs:ProfitLossFromOperatingActivities,1.286293e+09,COP,TrimestreAcumuladoActual,2021-01-01,2021-04-01,False,2021Q1
0,ifrs:Revenue,2.365046e+09,COP,TrimestreAcumuladoActual,2021-01-01,2021-04-01,False,2021Q1


In [17]:
def selected_value(period, tag):
    values = temporal_facts_df.loc[
        (temporal_facts_df["periodo"] == period)
        & (temporal_facts_df["tag"] == tag),
        "valor",
    ]
    return float(values.iloc[0]) if len(values) == 1 else np.nan

periods = [f"2021Q{quarter}" for quarter in range(1, 5)]
ytd_rows = []

for period in periods:
    ytd_rows.append({
        "periodo": period,
        "ingresos_ytd": selected_value(period, "ifrs:Revenue"),
        "finance_costs_ytd": selected_value(period, "ifrs:FinanceCosts"),
        "resultado_operativo_ytd": selected_value(period, "ifrs:ProfitLossFromOperatingActivities"),
        "d_and_a_ytd": selected_value(period, "ifrs:DepreciationAndAmortisationExpense"),
        "caja": selected_value(period, "ifrs:CashAndCashEquivalents"),
        "deuda_cp": selected_value(period, "ifrs:ShorttermBorrowings"),
        "deuda_lp": selected_value(period, "ifrs:LongtermBorrowings"),
    })

ytd_df = pd.DataFrame(ytd_rows).set_index("periodo")
print("Tabla temporal: flujos YTD y stocks al cierre")
display(ytd_df)

flow_ytd_columns = [
    "ingresos_ytd",
    "finance_costs_ytd",
    "resultado_operativo_ytd",
    "d_and_a_ytd",
]
flow_quarterly_df = ytd_df[flow_ytd_columns].diff()
flow_quarterly_df.iloc[0] = ytd_df.iloc[0][flow_ytd_columns]
flow_quarterly_df = flow_quarterly_df.rename(columns={
    "ingresos_ytd": "ingresos_trimestrales",
    "finance_costs_ytd": "finance_costs_trimestrales",
    "resultado_operativo_ytd": "resultado_operativo_trimestral",
    "d_and_a_ytd": "d_and_a_trimestral",
})
flow_quarterly_df["ebitda_trimestral"] = (
    flow_quarterly_df["resultado_operativo_trimestral"]
    + flow_quarterly_df["d_and_a_trimestral"]
)

print("Tabla de flujos trimestrales derivados por diferencias YTD")
display(flow_quarterly_df)

borrowings_ytd_df = pd.DataFrame(index=periods)
borrowings_ytd_df["deuda_total"] = ytd_df["deuda_cp"] + ytd_df["deuda_lp"]
borrowings_ytd_df["borrowings_reportado"] = [
    selected_value(period, "ifrs:Borrowings") for period in periods
]
borrowings_ytd_df["diferencia"] = (
    borrowings_ytd_df["borrowings_reportado"]
    - borrowings_ytd_df["deuda_total"]
)
print("Control de deuda total")
display(borrowings_ytd_df)

expected_counts = validation_counts_df["candidatos_consolidados"].eq(1).all()
all_debt_controls_ok = borrowings_ytd_df["diferencia"].eq(0).all()
all_ebitda_values_available = flow_quarterly_df["ebitda_trimestral"].notna().all()

print(f"Un candidato consolidado por tag y trimestre: {expected_counts}")
print(f"Borrowings = ShorttermBorrowings + LongtermBorrowings en Q1-Q4: {all_debt_controls_ok}")
print(f"EBITDA trimestral disponible en Q1-Q4: {all_ebitda_values_available}")

assert len(files_2021) == 4
assert expected_counts
assert all_debt_controls_ok
assert all_ebitda_values_available

Tabla temporal: flujos YTD y stocks al cierre


,ingresos_ytd,finance_costs_ytd,resultado_operativo_ytd,d_and_a_ytd,caja,deuda_cp,deuda_lp
periodo,,,,,,,
2021Q1,2.365046e+09,3.522148e+08,1.286293e+09,211194859.0,3.437159e+09,1.586526e+09,2.238512e+10
2021Q2,5.171052e+09,7.554423e+08,2.930900e+09,431842498.0,4.745206e+09,2.772660e+09,2.336138e+10
2021Q3,8.045883e+09,1.586391e+09,4.531038e+09,671091908.0,4.967287e+09,3.029718e+09,2.390651e+10
2021Q4,1.111694e+10,2.104483e+09,6.087393e+09,904840124.0,4.686462e+09,2.866267e+09,2.507418e+10


Tabla de flujos trimestrales derivados por diferencias YTD


,ingresos_trimestrales,finance_costs_trimestrales,resultado_operativo_trimestral,d_and_a_trimestral,ebitda_trimestral
periodo,,,,,
2021Q1,2.365046e+09,352214757.0,1.286293e+09,211194859.0,1.497488e+09
2021Q2,2.806006e+09,403227513.0,1.644607e+09,220647639.0,1.865254e+09
2021Q3,2.874831e+09,830948449.0,1.600138e+09,239249410.0,1.839388e+09
2021Q4,3.071054e+09,518092046.0,1.556356e+09,233748216.0,1.790104e+09


Control de deuda total


,deuda_total,borrowings_reportado,diferencia
2021Q1,2.397164e+10,2.397164e+10,0.0
2021Q2,2.613404e+10,2.613404e+10,0.0
2021Q3,2.693623e+10,2.693623e+10,0.0
2021Q4,2.794044e+10,2.794044e+10,0.0


Un candidato consolidado por tag y trimestre: True
Borrowings = ShorttermBorrowings + LongtermBorrowings en Q1-Q4: True
EBITDA trimestral disponible en Q1-Q4: True
